In [ ]:
# =============================================================================
# Zelle 01 – Environment Check: GPU, CUDA, Bibliotheken, Datenpfade
# =============================================================================

import os
import torch
import pandas as pd
import numpy as np

# GPU-Check
print("=== GPU Status ===")
print(f"CUDA verfuegbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"GPU-Speicher:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"Anzahl GPUs:    {torch.cuda.device_count()}")

# Datenpfade
print("\n=== Verfuegbare Dateien ===")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Datensatz laden
print("\n=== Datensatz laden ===")
# Pfad anpassen falls abweichend
TRAIN_PATH = '/kaggle/input/datasets/awatekoete/train-preprocessed/train_preprocessed.csv'
TEST_PATH = '/kaggle/input/datasets/awatekoete/test-test/test.csv'

train = pd.read_csv(TRAIN_PATH, keep_default_na=False)
train[["keyword", "location"]] = train[["keyword", "location"]].replace("", float("nan"))

test = pd.read_csv(TEST_PATH, keep_default_na=False)
test[["keyword", "location"]] = test[["keyword", "location"]].replace("", float("nan"))

print(f"Train Shape: {train.shape}")
print(f"Test Shape:  {test.shape}")
print(f"\nKlassenverteilung:")
print(train["target"].value_counts())
print(f"Disaster-Rate: {train['target'].mean()*100:.1f}%")
print(f"\nSpalten Train: {list(train.columns)}")
print(f"\nSpalten Test: {list(test.columns)}")

In [ ]:
# =============================================================================
# Zelle 02 – Bibliotheken installieren und importieren
# =============================================================================

# HuggingFace Transformers + Datasets (auf Kaggle meist vorinstalliert,
# aber explizit sicherstellen)
import subprocess
subprocess.run(["pip", "install", "transformers", "datasets",
                "accelerate", "-q"], check=True)

import torch
import numpy as np
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, roc_auc_score, matthews_corrcoef,
    precision_score, recall_score
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Transformers Version: "
      f"{__import__('transformers').__version__}")
print(f"PyTorch Version: {torch.__version__}")

# Reproduzierbarkeit
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# =============================================================================
# Zelle 03 – Preprocessing + DistilBERT Tokenizer laden
# =============================================================================
# Wichtig: Transformer brauchen den ORIGINAL-Tweet-Text (nicht gestemmt/lemmatisiert)
# Der Tokenizer macht sein eigenes Subword-Splitting (WordPiece)
# Wir nutzen die 'text'-Spalte aus train_preprocessed.csv (bereinigt,
# aber nicht gestemmt)

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128  # Tweets sind kurz, 128 reicht (max. Twitter: 280 Zeichen)

print(f"Lade Tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Test: wie tokenisiert DistilBERT einen Beispiel-Tweet?
sample_tweet = "forest fire near La Ronge Sask Canada"
tokens = tokenizer(sample_tweet, max_length=MAX_LENGTH,
                   truncation=True, padding="max_length")

print(f"\nBeispiel-Tweet: '{sample_tweet}'")
print(f"Token IDs:      {tokens['input_ids'][:15]}...")
print(f"Tokens:         {tokenizer.convert_ids_to_tokens(tokens['input_ids'])[:15]}...")
print(f"Anzahl Tokens:  {sum(1 for t in tokens['attention_mask'] if t == 1)}")

# Daten vorbereiten
texts = train["text"].fillna("").tolist()
labels = train["target"].tolist()

print(f"\nTrainingsdaten vorbereitet:")
print(f"  Anzahl Tweets: {len(texts)}")
print(f"  Klasse 0:      {labels.count(0)}")
print(f"  Klasse 1:      {labels.count(1)}")

In [ ]:
# =============================================================================
# Zelle 04 – Dataset erstellen + vollstaendige Metriken-Funktion
# =============================================================================
# Option B: 1 Split (80/20, stratified, Seed=42) x 4 Modelle
# Metriken: identisch mit Standardprojekt (F1, ROC-AUC, MCC, P1, R1)
# + Bootstrap-KI (95%, n=500) auf Validation-Set nach Training

from datasets import Dataset
from sklearn.model_selection import train_test_split

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )

def compute_metrics(eval_pred):
    """Metriken identisch mit Standardprojekt Phase 4/5."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = torch.softmax(
        torch.tensor(logits, dtype=torch.float32), dim=-1
    )[:, 1].numpy()

    # Bootstrap-KI (95%, n=500) auf Validation-Predictions
    rng = np.random.RandomState(SEED)
    boot_f1, boot_roc = [], []
    for _ in range(500):
        idx = rng.choice(len(labels), len(labels), replace=True)
        if len(np.unique(np.array(labels)[idx])) < 2:
            continue
        boot_f1.append(f1_score(
            np.array(labels)[idx],
            predictions[idx], average="macro"))
        boot_roc.append(roc_auc_score(
            np.array(labels)[idx], probs[idx]))

    return {
        "f1_macro":    round(f1_score(labels, predictions,
                                      average="macro"), 4),
        "f1_0":        round(f1_score(labels, predictions,
                                      pos_label=0), 4),
        "f1_1":        round(f1_score(labels, predictions,
                                      pos_label=1), 4),
        "precision_1": round(precision_score(labels, predictions,
                                             zero_division=0), 4),
        "recall_1":    round(recall_score(labels, predictions,
                                          zero_division=0), 4),
        "roc_auc":     round(roc_auc_score(labels, probs), 4),
        "mcc":         round(matthews_corrcoef(labels, predictions), 4),
        "f1_ci_low":   round(np.percentile(boot_f1, 2.5), 4),
        "f1_ci_high":  round(np.percentile(boot_f1, 97.5), 4),
        "roc_ci_low":  round(np.percentile(boot_roc, 2.5), 4),
        "roc_ci_high": round(np.percentile(boot_roc, 97.5), 4),
    }

# Train/Val Split (80/20, stratified)
X_train, X_val, y_train, y_val = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels,
)

print(f"Train: {len(X_train)} | Val: {len(X_val)}")
print(f"Train Disaster-Rate: {np.mean(y_train)*100:.1f}%")
print(f"Val   Disaster-Rate: {np.mean(y_val)*100:.1f}%")

# HuggingFace Datasets erstellen + tokenisieren
train_dataset = Dataset.from_dict({"text": X_train, "label": y_train})
val_dataset   = Dataset.from_dict({"text": X_val,   "label": y_val})

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function,   batched=True)

train_dataset.set_format(
    type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(
    type="torch", columns=["input_ids", "attention_mask", "label"])

print(f"\nDataset tokenisiert:")
print(f"  Train: {len(train_dataset)} Samples")
print(f"  Val:   {len(val_dataset)} Samples")
print(f"  Features: {list(train_dataset.features.keys())}")

In [ ]:
# =============================================================================
# Zelle 05 – DistilBERT Fine-Tuning (Stufe 1 von 4)
# =============================================================================

import os
from transformers import TrainingArguments, Trainer

# Modell laden (mit Klassifikations-Kopf fuer 2 Klassen)
print(f"Lade Modell: {MODEL_NAME}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "KEIN_DISASTER", 1: "DISASTER"},
    label2id={"KEIN_DISASTER": 0, "DISASTER": 1},
)
model.to(DEVICE)

print(f"Modell-Parameter: {model.num_parameters():,}")
print(f"Trainierbare Parameter: "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Training-Konfiguration
OUTPUT_DIR = "/kaggle/working/distilbert"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    fp16=True,               # Mixed Precision fuer T4 GPU
    dataloader_num_workers=2,
    report_to="none",        # kein Wandb/MLflow
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("\nStarte Training...")
print(f"  Epochen:    {training_args.num_train_epochs}")
print(f"  Batch-Size: {training_args.per_device_train_batch_size}")
print(f"  LR:         {training_args.learning_rate}")
print(f"  FP16:       {training_args.fp16}")
print(f"  Geschaetzte Laufzeit: ~15-25 Min auf T4")

train_result = trainer.train()

print("\n=== Training abgeschlossen ===")
print(f"  Trainingszeit: {train_result.metrics['train_runtime']:.0f}s "
      f"({train_result.metrics['train_runtime']/60:.1f} Min)")
print(f"  Train Loss:    {train_result.metrics['train_loss']:.4f}")

In [ ]:
# =============================================================================
# Zelle 06 – DistilBERT: finale Evaluierung + Ergebnisse speichern
# =============================================================================

import json
from sklearn.metrics import confusion_matrix

# Baseline-Referenzwerte (Quelle: models/champion_metadata.json, Phase 5)
# Hart kodiert weil champion_metadata.json nicht auf Kaggle verfuegbar —
# Werte stammen aus 5-Fold Stratified CV, Out-of-Fold (Notebook 05)
BASELINE = {
    "modell":       "TF-IDF + LogReg_balanced (Standard, Phase 5)",
    "f1_macro":     0.7775,
    "roc_auc":      0.8487,
    "mcc":          0.5551,
    "precision_1":  0.741,
    "recall_1":     0.728,
    "fnr":          0.272,
    "fpr":          0.174,
}

# Finale Evaluierung auf Validation-Set
print("Finale Evaluierung DistilBERT...")
eval_results = trainer.evaluate()

# Confusion Matrix
predictions_output = trainer.predict(val_dataset)
preds = np.argmax(predictions_output.predictions, axis=-1)
probs = torch.softmax(
    torch.tensor(predictions_output.predictions,
                 dtype=torch.float32), dim=-1)[:, 1].numpy()
true_labels = predictions_output.label_ids

cm = confusion_matrix(true_labels, preds)
tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp)
fpr = fp / (fp + tn)

# DistilBERT Ergebnisse
distilbert_results = {
    "modell":       "distilbert-base-uncased (Bonus, Phase 8)",
    "f1_macro":     eval_results["eval_f1_macro"],
    "roc_auc":      eval_results["eval_roc_auc"],
    "mcc":          eval_results["eval_mcc"],
    "precision_1":  eval_results["eval_precision_1"],
    "recall_1":     eval_results["eval_recall_1"],
    "f1_ci_low":    eval_results["eval_f1_ci_low"],
    "f1_ci_high":   eval_results["eval_f1_ci_high"],
    "fnr":          round(fnr, 4),
    "fpr":          round(fpr, 4),
    "tn": int(tn), "fp": int(fp),
    "fn": int(fn), "tp": int(tp),
    "epochs":       3,
    "learning_rate": 2e-5,
    "trainingszeit_sek": round(
        train_result.metrics["train_runtime"], 0),
}

# Ausgabe
print("\n=== DistilBERT Ergebnisse ===")
for k, v in eval_results.items():
    if not k.startswith("eval_runtime") and not k.startswith("eval_samples"):
        print(f"  {k:25s}: {v:.4f}")

print(f"\n=== Confusion Matrix ===")
print(f"  TN={tn:4d}  FP={fp:4d}")
print(f"  FN={fn:4d}  TP={tp:4d}")
print(f"  FNR: {fnr*100:.1f}% verpasste Disasters "
      f"(Baseline: {BASELINE['fnr']*100:.1f}%)")
print(f"  FPR: {fpr*100:.1f}% Fehlalarme "
      f"(Baseline: {BASELINE['fpr']*100:.1f}%)")

# Direkter Vergleich
print(f"\n=== Vergleich: Baseline vs. DistilBERT ===")
metrics = ["f1_macro", "roc_auc", "mcc", "precision_1", "recall_1",
           "fnr", "fpr"]
print(f"  {'Metrik':15s} | {'Baseline':10s} | {'DistilBERT':10s} | Delta")
print(f"  {'-'*55}")
for m in metrics:
    b = BASELINE[m]
    d = distilbert_results[m]
    delta = d - b
    sign = "+" if delta > 0 else ""
    better = "✅" if (delta > 0 and m not in ["fnr", "fpr"]) or \
             (delta < 0 and m in ["fnr", "fpr"]) else "❌"
    print(f"  {m:15s} | {b:10.4f} | {d:10.4f} | {sign}{delta:.4f} {better}")

# Speichern
os.makedirs("/kaggle/working/results", exist_ok=True)

comparison_df = pd.DataFrame([BASELINE, distilbert_results])
comparison_df.to_csv(
    "/kaggle/working/results/distilbert_vs_baseline.csv", index=False)

with open("/kaggle/working/results/distilbert_metadata.json", "w") as f:
    json.dump(distilbert_results, f, indent=2)

print("\nGespeichert: /kaggle/working/results/")

## Erkenntnis: DistilBERT Fine-Tuning (Stufe 1)

**Modell:** distilbert-base-uncased
**Training:** 3 Epochen, lr=2e-5, batch=32, FP16, T4×2
**Trainingszeit:** 107 Sekunden

### Ergebnisse (Val-Set, 1.361 Samples)

| Metrik | Baseline (LogReg) | DistilBERT | Delta |
|---|---|---|---|
| F1 Macro | 0.7775 | **0.8079** | +0.030 ✅ |
| ROC-AUC | 0.8487 | **0.8783** | +0.030 ✅ |
| MCC | 0.5551 | **0.6186** | +0.064 ✅ |
| Precision_1 | 0.741 | **0.806** | +0.065 ✅ |
| Recall_1 | 0.728 | 0.727 | -0.001 ❌ |
| FPR | 17.4% | **12.0%** | -5.4% ✅ |
| FNR | 27.2% | 27.3% | +0.1% ❌ |
| 95% KI F1 | [0.768-0.788] | [0.787-0.827] | ↑ |

### Kernbefunde

**Was sich verbessert hat:**
DistilBERT reduziert FP drastisch (-44% relativ: 703→97 auf Val-Set-Groesse
skaliert). Das FP-Problem aus Phase 06 (Metaphern, Alltagssprache mit
Disaster-Vokabular) wird durch Kontextverstaendnis teilweise geloest.

**Was sich NICHT verbessert hat:**
FNR praktisch unveraendert (27.2% → 27.3%). Das Kern-Problem aus Phase 06
(91.1% der FN haben kein Disaster-Vokabular) ist mit DistilBERT-base
nicht geloest. Recall_1 identisch mit Baseline.

**Erklaerung:**
DistilBERT-base wurde auf allgemeinen englischen Texten vortrainiert —
Twitter-spezifische Sprache (Slang, Abkuerzungen, Hashtags, @-Mentions)
ist im Vortraining unterrepraesentiert. Das FN-Problem erfordert moeglicherweise
Twitter-spezifisches Wissen.

**Hypothesis fuer BERTweet:**
BERTweet (850M Tweets vortrainiert) koennte das FN-Problem besser adressieren —
Twitter-Slang und kontextuelle Tweet-Semantik sind im Vortraining enthalten.
Erwartung: Recall_1 steigt, FNR sinkt unter 20%.

### Overfitting-Signal
Val Loss stagniert ab Epoch 2 (0.845→0.846) waehrend Train Loss weiter sinkt
(0.711→0.630) — leichtes Overfitting. 3 Epochen war richtige Wahl.

### Einordnung
DistilBERT ist ein echter Schritt vorwaerts (+3.0% F1) — aber es loest das
falsche Problem (FP statt FN). Der eigentlich interessante Vergleich folgt
mit BERTweet.

In [ ]:
# =============================================================================
# Zelle 08 – BERTweet Fine-Tuning (Stufe 2 von 4)
# =============================================================================
# BERTweet: auf 850M englischen Tweets vortrainiert (Nguyen et al. 2020)
# Twitter-spezifische Tokenisierung: @USER, HTTPURL, Emojis
# Hypothesis: besseres FNR als DistilBERT durch Twitter-Vorwissen
 
import gc, re
import torch

# Speicher freigeben (nur falls Vorgaenger-Modell noch im Kernel)
for var in ["model", "trainer"]:
    if var in dir():
        exec(f"del {var}")
gc.collect()
torch.cuda.empty_cache()

# GPU-Verifikation
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"Freier Speicher: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB")

BERTWEET_NAME = "vinai/bertweet-base"
MAX_LENGTH_BT = 128

# BERTweet-Preprocessing
def preprocess_for_bertweet(text):
    text = re.sub(r"@\w+", "@USER", str(text))
    text = re.sub(r"http\S+", "HTTPURL", text)
    return text

texts_bt = train["text"].fillna("").apply(preprocess_for_bertweet).tolist()

print(f"\nLade BERTweet Tokenizer: {BERTWEET_NAME}")
tokenizer_bt = AutoTokenizer.from_pretrained(BERTWEET_NAME)

# Beispiel
sample = "Check @elonmusk fire near Canada https://t.co/abc123"
print(f"Original:  {sample}")
print(f"BERTweet:  {preprocess_for_bertweet(sample)}")

# Gleicher Split wie DistilBERT (Seed=42 garantiert identische Aufteilung)
X_train_bt, X_val_bt, y_train_bt, y_val_bt = train_test_split(
    texts_bt, labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels,
)

def tokenize_bt(examples):
    return tokenizer_bt(
        examples["text"],
        max_length=MAX_LENGTH_BT,
        truncation=True,
        padding="max_length",
    )

train_dataset_bt = Dataset.from_dict({"text": X_train_bt, "label": y_train_bt})
val_dataset_bt   = Dataset.from_dict({"text": X_val_bt,   "label": y_val_bt})

train_dataset_bt = train_dataset_bt.map(tokenize_bt, batched=True)
val_dataset_bt   = val_dataset_bt.map(tokenize_bt,   batched=True)

train_dataset_bt.set_format(
    type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset_bt.set_format(
    type="torch", columns=["input_ids", "attention_mask", "label"])

# Modell laden
print(f"\nLade Modell: {BERTWEET_NAME}")
model_bt = AutoModelForSequenceClassification.from_pretrained(
    BERTWEET_NAME,
    num_labels=2,
    id2label={0: "KEIN_DISASTER", 1: "DISASTER"},
    label2id={"KEIN_DISASTER": 0, "DISASTER": 1},
)
model_bt.to(DEVICE)
print(f"Parameter: {model_bt.num_parameters():,}")

# Training
OUTPUT_DIR_BT = "/kaggle/working/bertweet"

training_args_bt = TrainingArguments(
    output_dir=OUTPUT_DIR_BT,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
)

trainer_bt = Trainer(
    model=model_bt,
    args=training_args_bt,
    train_dataset=train_dataset_bt,
    eval_dataset=val_dataset_bt,
    compute_metrics=compute_metrics,
)

print("\nStarte BERTweet Training...")
print("Geschaetzte Laufzeit: ~2-5 Min auf T4×2")
train_result_bt = trainer_bt.train()

print(f"\n=== Training abgeschlossen ===")
print(f"  Zeit:       {train_result_bt.metrics['train_runtime']:.0f}s "
      f"({train_result_bt.metrics['train_runtime']/60:.1f} Min)")
print(f"  Train Loss: {train_result_bt.metrics['train_loss']:.4f}")

## Erkenntnis: BERTweet Fine-Tuning (Stufe 2, erste Runde)

**Modell:** vinai/bertweet-base (RoBERTa-Architektur, 134M Parameter)
**Vortraining:** 850M englische Tweets (Twitter-spezifisch)
**Training:** 3 Epochen, lr=2e-5, batch=32, FP16, T4×2
**Trainingszeit:** 220s (3.7 Min)

### Epoch-Verlauf

| Epoch | Train Loss | Val Loss | F1 Macro | MCC |
|---|---|---|---|---|
| 1 | 1.334 | 0.879 | 0.807 | 0.618 |
| 2 | 0.797 | 0.933 | 0.791 | 0.586 |
| 3 | 0.669 | **0.827** | **0.825** | **0.656** |

**Instabiler Verlauf in Epoch 2:** Val Loss steigt (0.879→0.933)
bevor er in Epoch 3 wieder faellt (0.827). Kein Plateau erreicht —
das Modell konvergiert noch nicht.

### Vergleich: Baseline vs. DistilBERT vs. BERTweet

| Metrik | LogReg | DistilBERT | BERTweet | Δ vs. LogReg |
|---|---|---|---|---|
| F1 Macro | 0.7775 | 0.8079 | **0.8249** | +0.047 |
| ROC-AUC | 0.8487 | 0.8783 | **0.8785** | +0.030 |
| MCC | 0.5551 | 0.6186 | **0.6559** | +0.101 |
| Precision_1 | 0.741 | 0.806 | **0.846** | +0.105 |
| Recall_1 | 0.728 | 0.727 | 0.727 | -0.001 |
| FNR | 27.2% | 27.3% | ~27.3% | ~0% |

### Kritischer Befund: Recall_1 unveraendert

Alle drei Modelle zeigen Recall_1 ≈ 0.727 — der FNR-Befund aus
Phase 06 (91.1% der FN ohne Disaster-Vokabular) ist auch mit
Twitter-spezifischem Vortraining nicht geloest. BERTweet verbessert
Precision stark (+0.105 vs. Baseline), aber Recall nicht.

**Hypothese:** Das Problem liegt nicht im Modell oder Vortraining,
sondern im Datensatz selbst — Label-Rauschen und inhaerent ambige
Tweets setzen eine strukturelle Obergrenze fuer Recall_1.

### Warum BERTweet mehr Epochen braucht

| Indikator | DistilBERT | BERTweet |
|---|---|---|
| Val Loss Plateau | Ja (Ep2→Ep3: +0.001) | Nein (Ep2→Ep3: -0.106) |
| Train Loss Ende | 0.630 | 0.897 (noch hoch) |
| F1-Gewinn Ep1→Ep3 | +0.003 | +0.018 |
| Konvergiert? | Ja | Nein |

BERTweet hat mehr Parameter (134M vs. 67M) und tieferes
Twitter-spezifisches Vorwissen — es braucht mehr Schritte um
die Klassifikationsaufgabe zu internalisieren.

**Entscheidung:** BERTweet mit 2 zusaetzlichen Epochen (lr=1e-5,
stabiler) nachtrainieren — geringere LR reduziert das Instabilitaets-
Risiko (Epoch-2-Effekt) und ermoeglicht feineres Lernen.

In [ ]:
# =============================================================================
# Zelle 10 – BERTweet Nachtraining: 2 zusaetzliche Epochen, lr=1e-5
# =============================================================================
# Begruendung: Val Loss in Epoch 3 noch sinkend (0.827), kein Plateau.
# Kleinere LR (1e-5 statt 2e-5) reduziert Instabilitaet (Epoch-2-Effekt).

OUTPUT_DIR_BT2 = "/kaggle/working/bertweet_continued"

training_args_bt2 = TrainingArguments(
    output_dir=OUTPUT_DIR_BT2,
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=50,           # weniger Warmup (kurzes Training)
    weight_decay=0.01,
    learning_rate=1e-5,        # halbierte LR fuer stabiles Feintuning
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
)

trainer_bt2 = Trainer(
    model=model_bt,            # bereits trainiertes BERTweet-Modell
    args=training_args_bt2,
    train_dataset=train_dataset_bt,
    eval_dataset=val_dataset_bt,
    compute_metrics=compute_metrics,
)

print("Starte BERTweet Nachtraining (2 Epochen, lr=1e-5)...")
train_result_bt2 = trainer_bt2.train()

print(f"\n=== Nachtraining abgeschlossen ===")
print(f"  Zeit:       {train_result_bt2.metrics['train_runtime']:.0f}s")
print(f"  Train Loss: {train_result_bt2.metrics['train_loss']:.4f}")

In [ ]:
# =============================================================================
# Zelle 11 – BERTweet: besten Checkpoint laden (Epoch 3, lr=2e-5)
# =============================================================================
# Nachtraining hat Overfitting gezeigt (Val Loss stieg 0.827->0.948).
# Bester Checkpoint: Epoch 3 aus erstem Training (F1=0.8249).

import json
from sklearn.metrics import confusion_matrix

# Besten Checkpoint explizit laden (Epoch 3 aus erstem Training)
BEST_CHECKPOINT = "/kaggle/working/bertweet/checkpoint-255"

print(f"Lade besten BERTweet-Checkpoint: {BEST_CHECKPOINT}")
model_bt_best = AutoModelForSequenceClassification.from_pretrained(
    BEST_CHECKPOINT)
model_bt_best.to(DEVICE)

# Evaluierung mit bestem Checkpoint
trainer_eval = Trainer(
    model=model_bt_best,
    args=TrainingArguments(
        output_dir="/kaggle/working/tmp",
        per_device_eval_batch_size=64,
        fp16=True,
        report_to="none",
    ),
    eval_dataset=val_dataset_bt,
    compute_metrics=compute_metrics,
)

print("Finale Evaluierung BERTweet (bester Checkpoint)...")
eval_bt = trainer_eval.evaluate()

# Confusion Matrix
pred_output = trainer_eval.predict(val_dataset_bt)
preds_bt = np.argmax(pred_output.predictions, axis=-1)
probs_bt = torch.softmax(
    torch.tensor(pred_output.predictions, dtype=torch.float32),
    dim=-1)[:, 1].numpy()
true_labels_bt = pred_output.label_ids

cm_bt = confusion_matrix(true_labels_bt, preds_bt)
tn, fp, fn, tp = cm_bt.ravel()
fnr = fn / (fn + tp)
fpr = fp / (fp + tn)

print("\n=== BERTweet Finale Ergebnisse (Epoch 3, lr=2e-5) ===")
for k, v in eval_bt.items():
    if not k.startswith("eval_runtime") and not k.startswith("eval_samples"):
        print(f"  {k:25s}: {v:.4f}")

print(f"\n=== Confusion Matrix ===")
print(f"  TN={tn:4d}  FP={fp:4d}")
print(f"  FN={fn:4d}  TP={tp:4d}")
print(f"  FNR: {fnr*100:.1f}%  FPR: {fpr*100:.1f}%")

# Baseline-Referenz
BASELINE = {
    "f1_macro": 0.7775, "roc_auc": 0.8487, "mcc": 0.5551,
    "precision_1": 0.741, "recall_1": 0.728,
    "fnr": 0.272, "fpr": 0.174,
}

# Gesamtvergleich
bertweet_results = {
    "modell": "BERTweet (vinai/bertweet-base, Ep3)",
    "f1_macro":    eval_bt["eval_f1_macro"],
    "roc_auc":     eval_bt["eval_roc_auc"],
    "mcc":         eval_bt["eval_mcc"],
    "precision_1": eval_bt["eval_precision_1"],
    "recall_1":    eval_bt["eval_recall_1"],
    "f1_ci_low":   eval_bt["eval_f1_ci_low"],
    "f1_ci_high":  eval_bt["eval_f1_ci_high"],
    "fnr": round(fnr, 4),
    "fpr": round(fpr, 4),
}

print(f"\n=== Gesamtvergleich ===")
print(f"  {'Metrik':15s} | {'Baseline':10s} | {'DistilBERT':10s} | {'BERTweet':10s}")
print(f"  {'-'*55}")
distilbert = {
    "f1_macro": 0.8079, "roc_auc": 0.8783, "mcc": 0.6186,
    "precision_1": 0.8056, "recall_1": 0.7269,
    "fnr": 0.273, "fpr": 0.120,
}
for m in ["f1_macro", "roc_auc", "mcc", "precision_1",
          "recall_1", "fnr", "fpr"]:
    print(f"  {m:15s} | {BASELINE[m]:10.4f} | "
          f"{distilbert[m]:10.4f} | {bertweet_results[m]:10.4f}")

# Speichern
os.makedirs("/kaggle/working/results", exist_ok=True)
with open("/kaggle/working/results/bertweet_metadata.json", "w") as f:
    json.dump(bertweet_results, f, indent=2)
print("\nGespeichert: /kaggle/working/results/bertweet_metadata.json")

## Erkenntnis: BERTweet Fine-Tuning — finale Ergebnisse

**Modell:** vinai/bertweet-base (RoBERTa, 134M Parameter)
**Bester Checkpoint:** Epoch 3 (lr=2e-5) — Nachtraining zeigte Overfitting
**Trainingszeit gesamt:** ~6 Min (3+2 Epochen)

### Finale Ergebnisse (bester Checkpoint)

| Metrik | Baseline (LogReg) | DistilBERT | BERTweet | Δ vs. Baseline |
|---|---|---|---|---|
| F1 Macro | 0.7775 | 0.8079 | **0.8249** | +0.047 |
| ROC-AUC | 0.8487 | 0.8783 | **0.8785** | +0.030 |
| MCC | 0.5551 | 0.6186 | **0.6559** | +0.105 |
| Precision_1 | 0.741 | 0.806 | **0.846** | +0.105 |
| Recall_1 | 0.728 | 0.727 | 0.727 | -0.001 |
| FPR | 17.4% | 12.0% | **9.0%** | -8.4% |
| FNR | 27.2% | 27.3% | 27.3% | +0.1% |
| 95% KI F1 | — | [0.787-0.827] | **[0.804-0.847]** | ↑ |

### Nachtraining-Erkenntnis

Versuch: 2 zusaetzliche Epochen mit lr=1e-5.
Ergebnis: Val Loss stieg (0.827→0.878→0.948) — Overfitting.
Epoch 3 (lr=2e-5) war bereits das globale Maximum.
Lesson learned: ein scheinbar "nicht konvergiertes" Modell
kann trotzdem bereits sein Optimum erreicht haben. Der
Val-Loss-Anstieg in Epoch 2 war eine LR-bedingte Instabilitaet,
kein Zeichen von unfertigem Lernen.

### Haertester Befund: FNR strukturell unveraenderlich

Alle drei Modelle zeigen identische FNR (~27.3%):
- Baseline (TF-IDF): FNR = 27.2%
- DistilBERT:        FNR = 27.3%
- BERTweet:          FNR = 27.3%

Das ist keine Modell-Schwaeche — es ist eine Datensatz-Grenze.
Die 753 verpassten Disasters (Phase 06) enthalten zu 91.1%
kein Disaster-Vokabular. Auch BERTweet mit 850M Tweet-Vortraining
kann diese inhaerent ambigen Tweets nicht zuverlaessig erkennen.

**Interpretation:** Die FNR-Grenze (~27%) ist vermutlich nahe
dem Bayes-Fehler dieses Datensatzes — die theoretische Untergrenze
die kein Algorithmus unterschreiten kann, da die Labels selbst
inkonsistent sind (Label-Rauschen, Kontext-Abhaengigkeit).

### Was BERTweet besser macht

FPR halbiert (17.4% → 9.0%): BERTweet erkennt Metaphern,
Alltagssprache und nicht-disaster-bezogene Verwendung von
Disaster-Vokabular deutlich besser als TF-IDF.
Das ist der Twitter-Kontext-Effekt des Vortrainings.

### Naechster Schritt: BERT-base (Stufe 3)

Vergleichspunkt: generisches BERT vs. Twitter-spezifisches BERTweet.
Erwartung: BERTweet bleibt Champion (Twitter-Domaenanpassung),
BERT-base leicht schlechter (kein Twitter-Vortraining).

In [ ]:
# =============================================================================
# Zelle 13 – BERT-base Fine-Tuning (Stufe 3 von 4)
# =============================================================================
# BERT-base: generisches Vortraining (Wikipedia + BookCorpus)
# Vergleichspunkt: generisch vs. Twitter-spezifisch (BERTweet)
# Erwartung: BERTweet bleibt Champion, BERT-base leicht schlechter

import gc
import torch

# Speicher freigeben
for obj in ["model_bt", "model_bt_best", "trainer_bt",
            "trainer_bt2", "trainer_eval"]:
    if obj in dir():
        exec(f"del {obj}")
gc.collect()
torch.cuda.empty_cache()
print(f"GPU frei: {torch.cuda.mem_get_info(0)[0]/1e9:.1f} GB")

BERT_NAME = "bert-base-uncased"

# BERT-base benutzt Standard-Text (keine @USER/HTTPURL Konversion noetig)
X_train_bert, X_val_bert, y_train_bert, y_val_bert = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels,
)

print(f"Lade Tokenizer: {BERT_NAME}")
tokenizer_bert = AutoTokenizer.from_pretrained(BERT_NAME)

def tokenize_bert(examples):
    return tokenizer_bert(
        examples["text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )

train_dataset_bert = Dataset.from_dict(
    {"text": X_train_bert, "label": y_train_bert})
val_dataset_bert = Dataset.from_dict(
    {"text": X_val_bert, "label": y_val_bert})

train_dataset_bert = train_dataset_bert.map(tokenize_bert, batched=True)
val_dataset_bert   = val_dataset_bert.map(tokenize_bert,   batched=True)

train_dataset_bert.set_format(
    type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset_bert.set_format(
    type="torch", columns=["input_ids", "attention_mask", "label"])

# Modell laden
print(f"\nLade Modell: {BERT_NAME}")
model_bert = AutoModelForSequenceClassification.from_pretrained(
    BERT_NAME,
    num_labels=2,
    id2label={0: "KEIN_DISASTER", 1: "DISASTER"},
    label2id={"KEIN_DISASTER": 0, "DISASTER": 1},
)
model_bert.to(DEVICE)
print(f"Parameter: {model_bert.num_parameters():,}")

# Training
OUTPUT_DIR_BERT = "/kaggle/working/bert_base"

training_args_bert = TrainingArguments(
    output_dir=OUTPUT_DIR_BERT,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
)

trainer_bert = Trainer(
    model=model_bert,
    args=training_args_bert,
    train_dataset=train_dataset_bert,
    eval_dataset=val_dataset_bert,
    compute_metrics=compute_metrics,
)

print("\nStarte BERT-base Training...")
print("Geschaetzte Laufzeit: ~3-5 Min auf T4x2")
train_result_bert = trainer_bert.train()

print(f"\n=== Training abgeschlossen ===")
print(f"  Zeit:       {train_result_bert.metrics['train_runtime']:.0f}s "
      f"({train_result_bert.metrics['train_runtime']/60:.1f} Min)")
print(f"  Train Loss: {train_result_bert.metrics['train_loss']:.4f}")

## Erkenntnis: BERT-base Fine-Tuning (Stufe 3)

**Modell:** bert-base-uncased (BERT-Architektur, 109M Parameter)
**Vortraining:** Wikipedia + BookCorpus (generisch, kein Twitter)
**Training:** 3 Epochen, lr=2e-5, batch=32, FP16, T4×2
**Trainingszeit:** 224s (3.7 Min)

### Epoch-Verlauf

| Epoch | Train Loss | Val Loss | F1 Macro | MCC |
|---|---|---|---|---|
| 1 | 1.302 | 0.859 | 0.816 | 0.632 |
| 2 | 0.801 | 0.864 | 0.806 | 0.613 |
| 3 | 0.612 | **0.850** | **0.817** | **0.636** |

### Vollstaendiger Vergleich (alle Metriken)

| Metrik | Baseline | DistilBERT | BERTweet | BERT-base |
|---|---|---|---|---|
| F1 Macro | 0.7775 | 0.8079 | **0.8249** | 0.8173 |
| ROC-AUC | 0.8487 | 0.8783 | **0.8785** | 0.8849 |
| MCC | 0.5551 | 0.6186 | **0.6559** | 0.6361 |
| Precision_1 | 0.741 | 0.806 | **0.846** | 0.807 |
| Recall_1 | 0.728 | 0.727 | 0.727 | 0.751 |
| FPR | 17.4% | 12.0% | **9.0%** | 11.6% |
| FNR | 27.2% | 27.3% | 27.3% | **24.9%** |
| 95% KI F1 | — | [0.787-0.827] | [0.804-0.847] | ausstehend |

**Wichtigster Befund: BERT-base hat den besten Recall_1 (0.751)
und niedrigsten FNR (24.9%) aller Modelle — zum ersten Mal
sinkt die FNR unter 27%.**

ROC-AUC: BERT-base (0.885) > BERTweet (0.879) > DistilBERT (0.878)
— bei Ranking-Qualitaet ist BERT-base sogar bestes Modell.

**Interpretation:**
BERTweet hat besseres F1/MCC/Precision — weniger Fehlalarme.
BERT-base hat besseren Recall/FNR — verpasst weniger Disasters.
Das sind zwei verschiedene Optimierungsziele:
- Ressourcen-optimiert (wenig FP): BERTweet
- Disaster-Response (wenig FN): BERT-base

### Hypothese bestaetigt + neue Erkenntnis

Twitter-Vortraining (BERTweet) verbessert Precision (+0.039
vs BERT-base) — erkennt Metaphern/Alltagssprache besser.
Generisches Vortraining (BERT-base) verbessert Recall (+0.024
vs BERTweet) — findet mehr echte Disasters im Text.

**Moegliche Erklaerung:** BERT-base hat durch Wikipedia/BookCorpus
breiteres semantisches Verstaendnis von Katastrophen-Kontext
(Nachrichtenartikel-Stil), waehrend BERTweet Twitter-spezifische
Signale besser filtert (Metaphern, Slang).

### Konsequenz fuer Champion-Wahl

Kein klarer Gewinner — anwendungsfallabhaengig:
- F1/MCC/Precision → BERTweet Champion
- Recall/FNR/ROC-AUC → BERT-base Champion
- Disaster-Response-Kontext (FN teurer als FP) → BERT-base
  waere die professionellere Wahl

In [ ]:
# =============================================================================
# Zelle 15 – BERT-base: finale Evaluierung + Confusion Matrix
# =============================================================================

import json
from sklearn.metrics import confusion_matrix

print("Finale Evaluierung BERT-base...")
eval_bert = trainer_bert.evaluate()

# Confusion Matrix
pred_output_bert = trainer_bert.predict(val_dataset_bert)
preds_bert = np.argmax(pred_output_bert.predictions, axis=-1)
probs_bert = torch.softmax(
    torch.tensor(pred_output_bert.predictions, dtype=torch.float32),
    dim=-1)[:, 1].numpy()
true_labels_bert = pred_output_bert.label_ids

cm_bert = confusion_matrix(true_labels_bert, preds_bert)
tn, fp, fn, tp = cm_bert.ravel()
fnr = fn / (fn + tp)
fpr = fp / (fp + tn)

print("\n=== BERT-base Finale Ergebnisse ===")
for k, v in eval_bert.items():
    if not k.startswith("eval_runtime") and not k.startswith("eval_samples"):
        print(f"  {k:25s}: {v:.4f}")

print(f"\n=== Confusion Matrix ===")
print(f"  TN={tn:4d}  FP={fp:4d}")
print(f"  FN={fn:4d}  TP={tp:4d}")
print(f"  FNR: {fnr*100:.1f}%  FPR: {fpr*100:.1f}%")

# Vollstaendiger Vergleich alle Modelle
BASELINE    = {"f1_macro": 0.7775, "roc_auc": 0.8487, "mcc": 0.5551,
               "precision_1": 0.741, "recall_1": 0.728,
               "fnr": 0.272, "fpr": 0.174}
DISTILBERT  = {"f1_macro": 0.8079, "roc_auc": 0.8783, "mcc": 0.6186,
               "precision_1": 0.8056, "recall_1": 0.7269,
               "fnr": 0.273, "fpr": 0.120}
BERTWEET    = {"f1_macro": 0.8249, "roc_auc": 0.8785, "mcc": 0.6559,
               "precision_1": 0.8463, "recall_1": 0.7269,
               "fnr": 0.273, "fpr": 0.090}
BERT_BASE   = {"f1_macro": eval_bert["eval_f1_macro"],
               "roc_auc":  eval_bert["eval_roc_auc"],
               "mcc":      eval_bert["eval_mcc"],
               "precision_1": eval_bert["eval_precision_1"],
               "recall_1": eval_bert["eval_recall_1"],
               "fnr": round(fnr, 4), "fpr": round(fpr, 4)}

print(f"\n=== Vollstaendiger Vergleich: alle Modelle ===")
print(f"  {'Metrik':15s} | {'Baseline':10s} | {'DistilBERT':10s} | "
      f"{'BERTweet':10s} | {'BERT-base':10s}")
print(f"  {'-'*65}")
for m in ["f1_macro", "roc_auc", "mcc", "precision_1",
          "recall_1", "fnr", "fpr"]:
    vals = [BASELINE[m], DISTILBERT[m], BERTWEET[m], BERT_BASE[m]]
    # Bester Wert markieren (niedrigster bei fnr/fpr, hoechster sonst)
    if m in ["fnr", "fpr"]:
        best_idx = vals.index(min(vals))
    else:
        best_idx = vals.index(max(vals))
    row = f"  {m:15s} |"
    for i, v in enumerate(vals):
        marker = " ★" if i == best_idx else "  "
        row += f" {v:8.4f}{marker}|"
    print(row)

# Speichern
bert_base_results = {
    "modell": "bert-base-uncased (Phase 8, Stufe 3)",
    **BERT_BASE,
    "f1_ci_low":  eval_bert["eval_f1_ci_low"],
    "f1_ci_high": eval_bert["eval_f1_ci_high"],
    "tn": int(tn), "fp": int(fp),
    "fn": int(fn), "tp": int(tp),
}
os.makedirs("/kaggle/working/results", exist_ok=True)
with open("/kaggle/working/results/bert_base_metadata.json", "w") as f:
    json.dump(bert_base_results, f, indent=2)
print("\nGespeichert: /kaggle/working/results/bert_base_metadata.json")

In [ ]:
# =============================================================================
# Zelle 16 – Claude API Setup + Verbindungstest
# =============================================================================

import subprocess
subprocess.run(["pip", "install", "anthropic", "-q"], check=True)

import anthropic
from kaggle_secrets import UserSecretsClient

# API Key aus Kaggle Secrets laden
user_secrets = UserSecretsClient()
ANTHROPIC_API_KEY = user_secrets.get_secret("ANTHROPIC_API_KEY")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Verbindungstest
test_response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=10,
    messages=[{"role": "user", "content": "Reply with OK only."}]
)
print(f"API Test: {test_response.content[0].text}")
print(f"Modell:   {test_response.model}")
print(f"Tokens:   Input={test_response.usage.input_tokens} "
      f"Output={test_response.usage.output_tokens}")

In [ ]:
# =============================================================================
# Zelle 17 – Few-Shot Prompt Design + Zero-Shot/Few-Shot Klassifikation
# =============================================================================
# Zero-Shot: kein Beispiel im Prompt
# Few-Shot:  10 ausgewaehlte Beispiele (5 Disaster, 5 kein Disaster)
# Modell: claude-haiku-4-5-20251001 (schnell, guenstig)

import time
import json

# 10 Few-Shot Beispiele aus Trainingsdaten (ausgewogen, repraesentativ)
# Auswahl: klare Faelle (keine Grenzfaelle) fuer stabiles Few-Shot-Lernen
FEW_SHOT_EXAMPLES = [
    # Disaster (target=1)
    {"text": "Forest fire near La Ronge Sask Canada",
     "label": 1, "reason": "echte Naturkatastrophe"},
    {"text": "Earthquake magnitude 6.2 hits central Italy killing 5",
     "label": 1, "reason": "konkretes Erdbeben mit Opfern"},
    {"text": "Residents asked to shelter in place after chemical spill",
     "label": 1, "reason": "echte Notfallsituation"},
    {"text": "Typhoon Soudelor kills 28 in China and Taiwan",
     "label": 1, "reason": "Taifun mit Todesopfern"},
    {"text": "Wildfire evacuation orders lifted for parts of California",
     "label": 1, "reason": "Waldbrand-Evakuierung"},
    # Kein Disaster (target=0)
    {"text": "I am so devastated my team lost the game",
     "label": 0, "reason": "emotionale Metapher, kein echtes Disaster"},
    {"text": "She is a bombshell in that dress",
     "label": 0, "reason": "Alltagsausdruck, kein Anschlag"},
    {"text": "This traffic is absolutely killing me",
     "label": 0, "reason": "Redewendung, kein echter Tod"},
    {"text": "The concert was absolutely on fire last night",
     "label": 0, "reason": "Slang fuer grossartig, kein Brand"},
    {"text": "I am drowning in homework right now",
     "label": 0, "reason": "Metapher fuer Ueberforderung"},
]

# System-Prompt
SYSTEM_PROMPT = """Du bist ein Klassifikationssystem fuer Katastrophen-Tweets.
Deine Aufgabe: Entscheide ob ein Tweet ueber eine ECHTE Katastrophe berichtet.

Antworte NUR mit einer einzigen Zahl:
1 = echter Katastrophen-Tweet (Naturkatastrophe, Unfall, Anschlag etc.)
0 = kein Katastrophen-Tweet (Metapher, Slang, Fiktion, Alltag)

Keine Erklaerung. Nur 0 oder 1."""

# Few-Shot Beispiele als Text formatieren
def build_few_shot_context():
    context = "Beispiele:\n\n"
    for ex in FEW_SHOT_EXAMPLES:
        label = "1 (Disaster)" if ex["label"] == 1 else "0 (Kein Disaster)"
        context += f"Tweet: \"{ex['text']}\"\nAntwort: {label}\n\n"
    return context

FEW_SHOT_CONTEXT = build_few_shot_context()

def classify_tweet(text, mode="zero_shot", max_retries=3):
    """
    Klassifiziert einen Tweet mit Claude.
    mode: 'zero_shot' oder 'few_shot'
    """
    if mode == "few_shot":
        user_content = f"{FEW_SHOT_CONTEXT}Tweet: \"{text}\"\nAntwort:"
    else:
        user_content = f"Tweet: \"{text}\"\nAntwort:"

    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=5,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_content}]
            )
            answer = response.content[0].text.strip()
            # Robustes Parsen: nur erste Ziffer extrahieren
            for char in answer:
                if char in ["0", "1"]:
                    return int(char)
            return -1  # Ungueltige Antwort
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return -1

# Test mit 3 Beispiel-Tweets
print("=== API Test: 3 Beispiel-Tweets ===")
test_cases = [
    ("Forest fire near La Ronge Sask Canada", 1),
    ("I am so devastated my team lost the game", 0),
    ("Earthquake magnitude 6.2 hits central Italy killing 5", 1),
]

for text, true_label in test_cases:
    pred_zero = classify_tweet(text, mode="zero_shot")
    pred_few  = classify_tweet(text, mode="few_shot")
    print(f"\nTweet:      {text[:60]}")
    print(f"True Label: {true_label}")
    print(f"Zero-Shot:  {pred_zero} {'✅' if pred_zero == true_label else '❌'}")
    print(f"Few-Shot:   {pred_few}  {'✅' if pred_few == true_label else '❌'}")

print("\nPrompt-Design bereit.")
print(f"Few-Shot Beispiele: {len(FEW_SHOT_EXAMPLES)}")
print(f"Zero-Shot Tokens/Tweet: ~20-30")
print(f"Few-Shot Tokens/Tweet:  ~200-250")

In [ ]:
# =============================================================================
# Zelle 18 – Claude Zero-Shot + Few-Shot auf vollstaendigem Val-Set (1.361 Tweets)
# =============================================================================
# Laufzeit: ~15-25 Min (Rate Limits: ~50 Requests/Min bei Haiku)
# Kosten: ~$0.50-1.00 fuer beide Varianten

import time
from tqdm import tqdm

# Val-Set aus dem gleichen Split wie Transformer (Seed=42)
# X_val und y_val sind noch im Kernel von Zelle 04
val_texts = X_val   # Original-Tweets (nicht BERTweet-preprocessed)
val_labels = y_val

print(f"Val-Set: {len(val_texts)} Tweets")
print(f"Disaster-Rate: {np.mean(val_labels)*100:.1f}%")
print(f"\nGeschaetzte Laufzeit:")
print(f"  Zero-Shot: ~{len(val_texts)/50:.0f} Min")
print(f"  Few-Shot:  ~{len(val_texts)/50:.0f} Min")
print(f"  Gesamt:    ~{len(val_texts)/50*2:.0f} Min")
print(f"\nStarte Zero-Shot Klassifikation...")

# Zero-Shot
zero_shot_preds = []
errors_zero = 0
for i, text in enumerate(tqdm(val_texts)):
    pred = classify_tweet(text, mode="zero_shot")
    if pred == -1:
        pred = 0  # Fallback bei Fehler
        errors_zero += 1
    zero_shot_preds.append(pred)
    # Rate Limit: kurze Pause alle 50 Requests
    if (i + 1) % 50 == 0:
        time.sleep(1.5)

print(f"Zero-Shot abgeschlossen. Fehler: {errors_zero}")

print(f"\nStarte Few-Shot Klassifikation...")

# Few-Shot
few_shot_preds = []
errors_few = 0
for i, text in enumerate(tqdm(val_texts)):
    pred = classify_tweet(text, mode="few_shot")
    if pred == -1:
        pred = 0
        errors_few += 1
    few_shot_preds.append(pred)
    if (i + 1) % 50 == 0:
        time.sleep(1.5)

print(f"Few-Shot abgeschlossen. Fehler: {errors_few}")

# Metriken berechnen
from sklearn.metrics import (f1_score, roc_auc_score,
    matthews_corrcoef, precision_score, recall_score,
    confusion_matrix)

def calc_metrics(true, pred, name):
    cm = confusion_matrix(true, pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        "modell": name,
        "f1_macro":    round(f1_score(true, pred, average="macro"), 4),
        "f1_1":        round(f1_score(true, pred, pos_label=1), 4),
        "precision_1": round(precision_score(true, pred,
                                             zero_division=0), 4),
        "recall_1":    round(recall_score(true, pred,
                                          zero_division=0), 4),
        "mcc":         round(matthews_corrcoef(true, pred), 4),
        "fnr":         round(fn/(fn+tp), 4),
        "fpr":         round(fp/(fp+tn), 4),
        "tn": int(tn), "fp": int(fp),
        "fn": int(fn), "tp": int(tp),
    }

metrics_zero = calc_metrics(val_labels, zero_shot_preds,
                             "Claude Haiku Zero-Shot")
metrics_few  = calc_metrics(val_labels, few_shot_preds,
                             "Claude Haiku Few-Shot")

print("\n=== Ergebnisse ===")
print(f"\n{'Metrik':15s} | {'Zero-Shot':10s} | {'Few-Shot':10s}")
print(f"{'-'*40}")
for m in ["f1_macro", "mcc", "precision_1", "recall_1",
          "fnr", "fpr"]:
    print(f"{m:15s} | {metrics_zero[m]:10.4f} | "
          f"{metrics_few[m]:10.4f}")

# Speichern
os.makedirs("/kaggle/working/results", exist_ok=True)
pd.DataFrame([metrics_zero, metrics_few]).to_csv(
    "/kaggle/working/results/claude_results.csv", index=False)

# Vorhersagen fuer spaetere Fehleranalyse
pd.DataFrame({
    "text": val_texts,
    "true_label": val_labels,
    "zero_shot_pred": zero_shot_preds,
    "few_shot_pred": few_shot_preds,
}).to_csv("/kaggle/working/results/claude_predictions.csv",
          index=False)

print("\nGespeichert: /kaggle/working/results/")

In [ ]:
# =============================================================================
# Zelle 19 – Alle Ergebnisse zusammenfuehren + speichern
# =============================================================================

all_results = pd.DataFrame([
    {"modell": "TF-IDF + LogReg (Baseline)",
     "f1_macro": 0.7775, "mcc": 0.5551,
     "precision_1": 0.741, "recall_1": 0.728,
     "fnr": 0.272, "fpr": 0.174,
     "trainingszeit_sek": 1, "parameter": 5838},
    {"modell": "DistilBERT",
     "f1_macro": 0.8079, "mcc": 0.6186,
     "precision_1": 0.8056, "recall_1": 0.7269,
     "fnr": 0.273, "fpr": 0.120,
     "trainingszeit_sek": 107, "parameter": 67_000_000},
    {"modell": "BERT-base",
     "f1_macro": 0.8173, "mcc": 0.6361,
     "precision_1": 0.8074, "recall_1": 0.7505,
     "fnr": 0.250, "fpr": 0.123,
     "trainingszeit_sek": 224, "parameter": 109_000_000},
    {"modell": "BERTweet (Champion)",
     "f1_macro": 0.8249, "mcc": 0.6559,
     "precision_1": 0.8463, "recall_1": 0.7269,
     "fnr": 0.273, "fpr": 0.090,
     "trainingszeit_sek": 220, "parameter": 134_000_000},
    {"modell": "Claude Haiku Zero-Shot",
     "f1_macro": metrics_zero["f1_macro"],
     "mcc": metrics_zero["mcc"],
     "precision_1": metrics_zero["precision_1"],
     "recall_1": metrics_zero["recall_1"],
     "fnr": metrics_zero["fnr"],
     "fpr": metrics_zero["fpr"],
     "trainingszeit_sek": 0, "parameter": 0},
    {"modell": "Claude Haiku Few-Shot",
     "f1_macro": metrics_few["f1_macro"],
     "mcc": metrics_few["mcc"],
     "precision_1": metrics_few["precision_1"],
     "recall_1": metrics_few["recall_1"],
     "fnr": metrics_few["fnr"],
     "fpr": metrics_few["fpr"],
     "trainingszeit_sek": 0, "parameter": 0},
])

print("=== Vollstaendiger Modellvergleich ===")
print(all_results[["modell", "f1_macro", "mcc",
                    "precision_1", "recall_1",
                    "fnr", "fpr"]].to_string(index=False))

all_results.to_csv(
    "/kaggle/working/results/all_models_comparison.csv", index=False)
print("\nGespeichert: all_models_comparison.csv")

## Erkenntnis: Claude Haiku Few-Shot Klassifikation (Stufe 4)

**Modell:** claude-haiku-4-5-20251001 (kein Training, kein Fine-Tuning)
**Varianten:** Zero-Shot (kein Beispiel) + Few-Shot (10 Beispiele)
**Laufzeit:** ~45 Min fuer 2 x 1.361 Tweets | Kosten: ~$0.30

### Ergebnisse

| Metrik | Zero-Shot | Few-Shot |
|---|---|---|
| F1 Macro | **0.793** | 0.768 |
| MCC | **0.624** | 0.584 |
| Precision_1 | **0.912** | 0.903 |
| Recall_1 | **0.602** | 0.555 |
| FNR | **39.8%** | 44.5% |
| FPR | **4.0%** | 4.1% |

### Kernbefund 1: Few-Shot schlechter als Zero-Shot

Kontra-intuitives Ergebnis: 10 Beispiele im Prompt verschlechtern
die Performance (F1: 0.793→0.768, Recall: 0.602→0.555).

**Erklaerung:** Die 10 Beispiele waren zu eindeutig (klare Disasters
vs. klare Metaphern). Claude hat gelernt: "nur bei sehr klaren
Faellen Disaster sagen" — das verschaerft die ohnehin konservative
Tendenz weiter.

**Lesson learned:** Few-Shot-Beispiele muessen repraesentativ fuer
Grenzfaelle sein, nicht nur fuer eindeutige Faelle. Ambige Tweets
wie "body bagging Meek" oder "she's a suicide bomb" als Beispiele
haetten Claude besser kalibriert.

### Kernbefund 2: Claude hat einzigartiges Fehler-Profil

| Aspekt | Fine-Tuned (BERTweet) | Claude Zero-Shot |
|---|---|---|
| FPR | 9.0% | **4.0%** (bester Wert) |
| FNR | 27.3% | **39.8%** (schlechtester Wert) |
| Charakter | Ausgewogen | Extrem konservativ |

Claude ist das praeziseste Modell (Precision_1=0.912) —
fast keine Fehlalarme. Aber es verpasst 40% aller echten Disasters.

**Warum:** Claude wurde nicht auf diesem Datensatz trainiert.
Es hat keine gelernten Schwellenwerte fuer "wie sicher muss ich
sein bevor ich Disaster sage". Bei Unsicherheit bevorzugt es
systematisch "kein Disaster" — das ist eine Modell-Eigenschaft,
keine Schwaeche des Prompts.

### Vollstaendiger Modellvergleich

| Modell | F1 | MCC | P1 | R1 | FNR | FPR |
|---|---|---|---|---|---|---|
| TF-IDF + LogReg | 0.778 | 0.555 | 0.741 | 0.728 | 27.2% | 17.4% |
| DistilBERT | 0.808 | 0.619 | 0.806 | 0.727 | 27.3% | 12.0% |
| BERT-base | 0.817 | 0.636 | 0.807 | 0.751 | 25.0% | 12.3% |
| BERTweet | **0.825** | **0.656** | 0.846 | 0.727 | 27.3% | **9.0%** |
| Claude Zero-Shot | 0.793 | 0.624 | **0.912** | 0.602 | 39.8% | 4.0% |
| Claude Few-Shot | 0.768 | 0.584 | 0.903 | 0.555 | 44.5% | 4.1% |

### Konsequenz fuer Anwendungsfall

| Szenario | Bestes Modell | Begruendung |
|---|---|---|
| Maximale Praezision (kein Fehlalarm) | **Claude Zero-Shot** | FPR 4.0% |
| Disaster-Response (kein Missed Event) | **BERT-base** | FNR 25.0% |
| Beste Gesamtperformance (F1/MCC) | **BERTweet** | F1=0.825 |
| Kein Training moeglich | **Claude Zero-Shot** | F1=0.793 ohne Training |
| Beste Erklaerbarkeit | TF-IDF + LogReg | Koeffizienten direkt lesbar |

### Theoretische Einordnung

Claude Zero-Shot (F1=0.793) liegt zwischen TF-IDF+LogReg (0.778) und
DistilBERT (0.808) — ohne ein einziges Trainingsbeispiel gesehen zu haben.
Das zeigt die Staerke grosser Sprachmodelle: allgemeines Sprachverstaendnis
reicht fuer solide Baseline-Performance. Aber Fine-Tuning (BERTweet: 0.825)
schlaegt Zero-Shot deutlich — der Datensatz-spezifische Kontext ist entscheidend.

In [ ]:
# =============================================================================
# Zelle 21 – Gesamtvisualisierung: alle Modelle im Vergleich
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Store44 Farben (konsistent mit Standardprojekt)
BG       = "#1C1C1C"
GOLD     = "#F5A623"
BLUE     = "#6CB4E4"
GREEN    = "#4A5C3A"
MUTED    = "#888888"
RED      = "#E05C5C"

# Daten aus all_results DataFrame
models = all_results["modell"].tolist()
models_short = [
    "TF-IDF\nLogReg",
    "DistilBERT",
    "BERT-base",
    "BERTweet\n★Champion",
    "Claude\nZero-Shot",
    "Claude\nFew-Shot",
]

# Farben pro Modell-Kategorie
model_colors = [MUTED, BLUE, BLUE, GOLD, GREEN, GREEN]

f1_vals  = all_results["f1_macro"].tolist()
mcc_vals = all_results["mcc"].tolist()
p1_vals  = all_results["precision_1"].tolist()
r1_vals  = all_results["recall_1"].tolist()
fnr_vals = all_results["fnr"].tolist()
fpr_vals = all_results["fpr"].tolist()

x = np.arange(len(models_short))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor(BG)

for ax in axes.flatten():
    ax.set_facecolor(BG)
    ax.tick_params(colors="white")
    ax.spines[:].set_color(MUTED)

def bar_plot(ax, values, title, ylabel, invert=False):
    bars = ax.bar(x, values, color=model_colors, alpha=0.85, width=0.6)
    ax.set_title(title, color="white", fontsize=11)
    ax.set_ylabel(ylabel, color="white", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(models_short, fontsize=8, color="white")
    # Besten Wert markieren
    if invert:
        best_idx = values.index(min(values))
    else:
        best_idx = values.index(max(values))
    bars[best_idx].set_edgecolor("white")
    bars[best_idx].set_linewidth(2)
    # Werte annotieren
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", color="white",
                fontsize=7)

bar_plot(axes[0,0], f1_vals,  "F1 Macro",    "F1")
bar_plot(axes[0,1], mcc_vals, "MCC",         "MCC")
bar_plot(axes[0,2], p1_vals,  "Precision_1", "Precision")
bar_plot(axes[1,0], r1_vals,  "Recall_1",    "Recall")
bar_plot(axes[1,1], fnr_vals, "FNR (↓ besser)", "FNR", invert=True)
bar_plot(axes[1,2], fpr_vals, "FPR (↓ besser)", "FPR", invert=True)

# Legende
legend_patches = [
    mpatches.Patch(color=MUTED,  label="Klassisch (TF-IDF)"),
    mpatches.Patch(color=BLUE,   label="Transformer (generisch)"),
    mpatches.Patch(color=GOLD,   label="Transformer (Twitter)"),
    mpatches.Patch(color=GREEN,  label="LLM (kein Training)"),
]
fig.legend(handles=legend_patches, loc="lower center",
           ncol=4, fontsize=9,
           facecolor=BG, labelcolor="white",
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle(
    "Bonus Phase 8: Vollstaendiger Modellvergleich — alle 6 Modelle\n"
    "Weisser Rahmen = bester Wert pro Metrik",
    color="white", fontsize=13, y=1.01)

fig.tight_layout()
plt.savefig("/kaggle/working/results/08_all_models_comparison.png",
            dpi=150, bbox_inches="tight",
            facecolor=BG, edgecolor="none")
plt.show()
print("Gespeichert: 08_all_models_comparison.png")

# Abschluss: Bonus Phase 8 — Transformer & LLM Vergleich

## Uebersicht

| Modell | Typ | Training | F1 | MCC | P1 | R1 | FNR | FPR |
|---|---|---|---|---|---|---|---|---|
| TF-IDF + LogReg | Klassisch | <1 Sek | 0.778 | 0.555 | 0.741 | 0.728 | 27.2% | 17.4% |
| DistilBERT | Transformer | 107 Sek | 0.808 | 0.619 | 0.806 | 0.727 | 27.3% | 12.0% |
| BERT-base | Transformer | 224 Sek | 0.817 | 0.636 | 0.807 | 0.751 | 25.0% | 12.3% |
| BERTweet ★ | Transformer | 220 Sek | **0.825** | **0.656** | **0.846** | 0.727 | 27.3% | **9.0%** |
| Claude Zero-Shot | LLM | 0 Sek | 0.793 | 0.624 | **0.912** | 0.602 | 39.8% | **4.0%** |
| Claude Few-Shot | LLM | 0 Sek | 0.768 | 0.584 | 0.903 | 0.555 | 44.5% | 4.1% |

## Kernerkenntnisse

**1) BERTweet ist Gesamtchampion (F1=0.825, MCC=0.656)**
Twitter-spezifisches Vortraining (850M Tweets) liefert den besten
Kompromiss aus Precision und Recall. Abstand zur Baseline: +0.047 F1.

**2) BERT-base loest das FNR-Problem am besten (FNR=25.0%)**
Einziges Modell das die FNR-Mauer (~27%) durchbricht. Breiteres
semantisches Vorwissen (Wikipedia/BookCorpus) erkennt Disaster-Kontext
auch ohne explizites Disaster-Vokabular besser als Twitter-spezifisches
Modell.

**3) Claude Zero-Shot hat einzigartiges Profil (FPR=4.0%, FNR=39.8%)**
Praezisestes Modell (Precision_1=0.912) — fast keine Fehlalarme.
Aber 40% aller echten Disasters verpasst. Ohne Fine-Tuning ist Claude
zu konservativ fuer Disaster-Response-Anwendungen.

**4) Few-Shot schlechter als Zero-Shot**
10 eindeutige Beispiele haben Claude noch konservativer gemacht.
Grenzfaelle als Few-Shot-Beispiele waeren besser gewesen.

**5) FNR-Mauer (~27%) — strukturelle Datensatz-Grenze**
TF-IDF, DistilBERT und BERTweet zeigen alle FNR ~27.3%.
91.1% der FN haben kein Disaster-Vokabular (Phase 06).
Das ist nahe dem Bayes-Fehler dieses Datensatzes — Label-Rauschen
und inhaerent ambige Tweets setzen eine theoretische Untergrenze.

**6) Diminishing Returns bei Modell-Komplexitaet**
| Schritt | Delta F1 | Zusatz-Parameter |
|---|---|---|
| TF-IDF → DistilBERT | +0.030 | 67M |
| DistilBERT → BERT-base | +0.009 | 42M |
| BERT-base → BERTweet | +0.008 | 25M |
Jede zusaetzliche Modell-Komplexitaet bringt weniger Gewinn.

## Anwendungsfall-Empfehlungen

| Szenario | Empfehlung | Begruendung |
|---|---|---|
| Disaster-Response (FN teuer) | **BERT-base** | Niedrigster FNR (25.0%) |
| Ressourcen-optimiert (FP teuer) | **Claude Zero-Shot** | Niedrigster FPR (4.0%) |
| Beste Gesamtperformance | **BERTweet** | F1=0.825, MCC=0.656 |
| Kein Training moeglich | **Claude Zero-Shot** | F1=0.793 ohne Training |
| Interpretierbarkeit | **TF-IDF + LogReg** | Koeffizienten direkt lesbar |
| Schnellste Loesung | **TF-IDF + LogReg** | <1 Sek Training |

## Datensatz-Variante (offen)

Geplante Ablation (bereinigt 6.801 vs. original 7.613) wurde nicht
durchgefuehrt — Zeitbudget. Erwartung basierend auf Literatur:
Delta < 0.005 F1 fuer vortrainierte Transformer (robust gegen Rauschen).
Bleibt als offener Punkt fuer weitere Forschung.

## SOTA-Einordnung (final)

| Referenz | F1 |
|---|---|
| Unser Standardprojekt (TF-IDF) | 0.778 |
| Unser Bonus-Champion (BERTweet) | **0.825** |
| Human-Level (geschaetzt) | ~0.800 |
| Realistischer SOTA (BERTweet-Large) | ~0.850 |
| Kaggle Top* (*Leakage-verzerrt) | 0.890 |

BERTweet (0.825) uebertrifft geschaetztes Human-Level (~0.800) —
das zeigt die Staerke Twitter-spezifischen Vortrainings. Der
verbleibende Abstand zu SOTA (~0.025) liegt in Datenmenge und
Modellgroesse (BERTweet-Large statt BERTweet-base).

## Gespeicherte Artefakte

| Datei | Inhalt |
|---|---|
| /kaggle/working/bertweet/checkpoint-255 | BERTweet Champion Checkpoint |
| /kaggle/working/results/all_models_comparison.csv | Alle Metriken |
| /kaggle/working/results/08_all_models_comparison.png | Visualisierung |
| /kaggle/working/results/claude_predictions.csv | Claude Vorhersagen |
| /kaggle/working/results/distilbert_metadata.json | DistilBERT Metriken |
| /kaggle/working/results/bertweet_metadata.json | BERTweet Metriken |
| /kaggle/working/results/bert_base_metadata.json | BERT-base Metriken |